In [6]:
function EMpath(u0,T,N,d,m,fhandle,ghandle,kappa0, M)
    #small step
    Dtref = T/N 
    #large step
    Dt = kappa0*Dtref

    #number of large steps
    NN = N/kappa0
    #initialization of solution
    u = zeros(d,M,NN+1)
    #initializaion of time grid
    t = zeros(NN+1,1)
    #initialization of gdW
    gdW=zeros(d,M)
    sqrtDtref = sqrt(Dtref)

    #current u, start with u0
    u_n = u0

    for n=1:1:NN+1
        t[n] = (n-1)*Dt
        u[:,:,n] = u_n
        dW = sqrtDtref*squeeze(sum(randn(m,M,kappa0),3))
        for mm=1:M
            gdW[:,mm]=ghandle(u_n[:,mm])*dW[:,mm]
        end
        u_new = u_n+Dt*fhandle(u_n)+gdW
        u_n = u_new
    end
    return t, u
end

EMpath (generic function with 1 method)

In [7]:
function runEMpath(u0,T,Nref,d,m,fhandle,ghandle,kappa,M)
    S = 0.0 #counter for sum
    Mstep = 1000 #blocksize
    m0 = 1 #indexing for the blocks

    for mm in 1:Mstep:M
        #num of samples in this block (not needed if M is a multiple of Mstep)
        MM = min(Mstep, M - mm + 1) 
        
        #initial values for MM samples
        u00 = u0[:, mm:m0+MM-1]

        #save rng state
        rng = Random.default_rng()
        savedState = copy(rng)

        #reference solution and time grid using EMpath function (Algorithm 8.5) 
        tref, uref = EMpath(u00, T, Nref, d, m, fhandle, ghandle, 1, MM)

        #restore the state so we get same sample path for both the reference and coarse solution
        Random.seed!(rng, savedState.seed)

        #coarse solution and time grid using EMpath function (Algortithm 8.5)
        t, u = EMpath(u00, T, Nref, d, m, fhandle, ghandle, kappa, MM)

        #error at the last time slice 
        err = u[:, :, end] .- uref[:, :, end]

        #add the sum of the elementwise squared errors to the counter
        S += sum(err .* err) 

        #set new indexing for m0 since we are moving on to next block
        m0 += MM
    end

    #operations to get RMS 
    rmsErr = sqrt(S / M) 

    return rmsErr, t, u, tref, uref
end
        
    

runEMpath (generic function with 1 method)